In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np

# ── Load ──────────────────────────────────────────────────────────────────────
comments = pd.read_csv('hn_claude_comments_cleaned_.csv')
stories  = pd.read_csv('hn_claude_stories_cleaned.csv')

comments['created_at'] = pd.to_datetime(comments['created_at'], utc=True)
stories['created_at']  = pd.to_datetime(stories['created_at'],  utc=True)
comments['month'] = comments['created_at'].dt.to_period('M')
stories['month']  = stories['created_at'].dt.to_period('M')

# ── CHART 1: Hockey Stick ─────────────────────────────────────────────────────
cmt_m   = comments.groupby('month').size().rename('comments')
story_m = stories.groupby('month').size().rename('stories')
monthly = pd.concat([cmt_m, story_m], axis=1).fillna(0).astype(int)
monthly.index = monthly.index.to_timestamp()

fig, ax = plt.subplots(figsize=(13, 6))
ax.fill_between(monthly.index, monthly['comments'], alpha=0.18, color='#79c0ff')
ax.plot(monthly.index, monthly['comments'], color='#79c0ff', lw=2.5, label='Comments', marker='o')
ax.plot(monthly.index, monthly['stories'],  color='#56d364', lw=2.5, label='Stories',  marker='s')
ax.plot(monthly.index, monthly.sum(axis=1), color='#f78166', lw=3, label='Total', linestyle='--')

# Annotate launch events
for date, label in [('2026-02-01','Claude 3.7 Sonnet'),
                    ('2026-03-01','Claude Code GA'),
                    ('2026-04-04','OpenClaw Ban + Opus 4.6')]:
    ax.axvline(pd.Timestamp(date), linestyle=':', alpha=0.8)
    ax.text(pd.Timestamp(date), 20, label, fontsize=8, ha='center')

ax.set_title('Monthly HN Activity: Stories + Comments'); ax.legend(); plt.tight_layout()

# ── CHART 2: Harness Ecosystem ────────────────────────────────────────────────
all_text = pd.concat([comments['comment_text'].fillna(''),
                      stories['title'].fillna('') + ' ' + stories['story_text'].fillna('')])
tools = {'Claude Code':'claude code','OpenClaw':'openclaw','Codex (OpenAI)':'codex',
         'Cursor':'cursor','GitHub Copilot':'copilot','MCP (Protocol)':' mcp ',
         'OpenCode':'opencode','Aider':'aider'}
counts = {k: all_text.str.lower().str.contains(v, regex=False).sum() for k,v in tools.items()}
# → bar chart sorted descending

# ── CHART 3: Sentiment Shift ──────────────────────────────────────────────────
pos_kw = ['amazing','productivity','love','great','best','autonom','replac','incredible','outperform']
neg_kw = ['limit','billing','security','slop','hallucin','ban','frustrat','expensive','restricted']
comments['pos'] = comments['comment_text'].apply(lambda x: sum(k in str(x).lower() for k in pos_kw))
comments['neg'] = comments['comment_text'].apply(lambda x: sum(k in str(x).lower() for k in neg_kw))
sent = comments.groupby('month')[['pos','neg']].sum()
sent.index = sent.index.to_timestamp()
sent_pct = sent.div(sent.sum(axis=1), axis=0) * 100
# → stackplot of pos% vs neg%

# ── CHART 4: Model vs Harness Pie ────────────────────────────────────────────
model_kw   = ['opus','sonnet','reasoning','context window','underlying model','base model','1m token']
harness_kw = ['claude code','mcp','agentic','workflow','orchestrat','loop','multi-agent','subagent']
model_score   = comments['comment_text'].fillna('').apply(lambda x: sum(k in x.lower() for k in model_kw)).sum()
harness_score = comments['comment_text'].fillna('').apply(lambda x: sum(k in x.lower() for k in harness_kw)).sum()
# model=288, harness=618 → pie: 32% model, 68% harness

